# 🥈 04 — Dhara: Silver Layer (Cleaned & Normalized)

Normalizes heterogeneous train-timings CSVs into a clean `silver.train_timings` schema, and builds a `silver.station_master` lookup used for joins with air quality.

The raw CSVs have varied column names (`Name of the Train`, `train_name`, `Train No.`, etc.). We handle this via dynamic column detection.

In [0]:
%sql
USE CATALOG setu_rail;
USE SCHEMA silver;

## Step 1 — Normalize train timings

In [0]:
from pyspark.sql.functions import col, trim, upper, coalesce, lit, regexp_extract, regexp_replace, when, hour, to_timestamp

bronze = spark.table("setu_rail.bronze.train_runs")

# Dynamic resolution — pick whichever name is present
cols = [c.lower() for c in bronze.columns]
def pick(*candidates):
    for cand in candidates:
        if cand.lower() in cols:
            return bronze.columns[cols.index(cand.lower())]
    return None

train_name_col = pick("Name of the Train", "train_name", "Train Name", "name_of_train")
train_no_col   = pick("Train No.", "train_no", "Train Number", "number")
time_col       = pick("Time (Towards Arakkonam)", "Time", "Arrival Time", "arrival_time", "departure_time", "Departure", "Arrival")

print(f"Resolved -> name:{train_name_col}  no:{train_no_col}  time:{time_col}")

silver_timings = bronze.select(
    col(train_name_col).alias("train_name") if train_name_col else lit(None).cast("string").alias("train_name"),
    (col(train_no_col).cast("string") if train_no_col else lit(None).cast("string")).alias("train_no"),
    col(time_col).alias("scheduled_time_raw") if time_col else lit(None).cast("string").alias("scheduled_time_raw"),
    col("_source_file").alias("source_file"),
)

# Parse time strings like "7.00 AM" or "14:30" into a scheduled_hour integer
silver_timings = (silver_timings
    .withColumn("time_ampm", regexp_extract(col("scheduled_time_raw"), r"(AM|PM|am|pm)", 1))
    .withColumn("time_digits", regexp_replace(col("scheduled_time_raw"), r"[^0-9\.:]", ""))
    .withColumn("hh", regexp_extract(col("time_digits"), r"(\d{1,2})", 1).cast("int"))
    .withColumn("scheduled_hour",
        when((col("time_ampm").rlike("(?i)PM")) & (col("hh") < 12), col("hh") + 12)
        .when((col("time_ampm").rlike("(?i)AM")) & (col("hh") == 12), lit(0))
        .otherwise(col("hh")))
    # Extract train_no from name if column missing
    .withColumn("train_no_from_name",
        regexp_extract(col("train_name"), r"(\d{4,5})", 1))
    .withColumn("train_no",
        when(col("train_no").isNull() | (col("train_no") == ""), col("train_no_from_name"))
        .otherwise(col("train_no")))
    .drop("time_ampm", "time_digits", "hh", "train_no_from_name")
)

# Synthesize a home-city for each train based on the name (used as fallback weather join key)
silver_timings = silver_timings.withColumn(
    "inferred_city",
    when(col("train_name").rlike("(?i)Chennai|Madras|Arakkonam|Tambaram"), lit("CHENNAI"))
    .when(col("train_name").rlike("(?i)Bangalore|Bengaluru"),              lit("BENGALURU"))
    .when(col("train_name").rlike("(?i)Tirupathi|Tirupati"),               lit("TIRUPATI"))
    .when(col("train_name").rlike("(?i)Hyderabad|Charminar"),              lit("HYDERABAD"))
    .when(col("train_name").rlike("(?i)Mumbai|Bombay"),                    lit("MUMBAI"))
    .when(col("train_name").rlike("(?i)Delhi|Rajdhani"),                   lit("DELHI"))
    .when(col("train_name").rlike("(?i)Kolkata|Howrah"),                   lit("KOLKATA"))
    .otherwise(lit("CHENNAI")))

(silver_timings.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("setu_rail.silver.train_timings"))

print(f"✅ silver.train_timings: {silver_timings.count()} rows")
silver_timings.show(10, truncate=False)

## Step 2 — Build station_master (top Indian stations with city mapping)

In [0]:
# Curated list of 20 major Indian rail stations mapped to their AQ cities.
# Real-world data: these are all real station codes.
stations = [
    ("MAS",  "Chennai Central",        "CHENNAI",       13.0827, 80.2707, "SR",  1),
    ("MS",   "Chennai Egmore",         "CHENNAI",       13.0780, 80.2610, "SR",  1),
    ("SBC",  "KSR Bengaluru",          "BENGALURU",     12.9780, 77.5700, "SWR", 1),
    ("HYB",  "Hyderabad Deccan",       "HYDERABAD",     17.3850, 78.4867, "SCR", 1),
    ("SC",   "Secunderabad Junction",  "HYDERABAD",     17.4356, 78.5031, "SCR", 1),
    ("CSMT", "Mumbai CSMT",            "MUMBAI",        18.9398, 72.8355, "CR",  1),
    ("BCT",  "Mumbai Central",         "MUMBAI",        18.9710, 72.8195, "WR",  1),
    ("NDLS", "New Delhi",              "DELHI",         28.6419, 77.2197, "NR",  1),
    ("DLI",  "Old Delhi",              "DELHI",         28.6667, 77.2290, "NR",  1),
    ("HWH",  "Howrah Junction",        "KOLKATA",       22.5830, 88.3420, "ER",  1),
    ("SDAH", "Sealdah",                "KOLKATA",       22.5675, 88.3700, "ER",  1),
    ("PUNE", "Pune Junction",          "PUNE",          18.5286, 73.8740, "CR",  1),
    ("JP",   "Jaipur Junction",        "JAIPUR",        26.9187, 75.7878, "NWR", 1),
    ("LKO",  "Lucknow Charbagh",       "LUCKNOW",       26.8310, 80.9230, "NR",  1),
    ("PNBE", "Patna Junction",         "PATNA",         25.6020, 85.1400, "ECR", 1),
    ("BBS",  "Bhubaneswar",            "BHUBANESWAR",   20.2700, 85.8400, "ECoR",1),
    ("TPTY", "Tirupati",               "TIRUPATI",      13.6288, 79.4192, "SCR", 1),
    ("AJJ",  "Arakkonam",              "CHENNAI",       13.0833, 79.6667, "SR",  0),
    ("KGP",  "Kharagpur",              "KHARAGPUR",     22.3460, 87.3220, "SER", 0),
    ("ED",   "Erode Junction",         "ERODE",         11.3410, 77.7172, "SR",  0),
]

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
stn_schema = StructType([
    StructField("station_code", StringType(), False),
    StructField("station_name", StringType(), False),
    StructField("city",         StringType(), False),
    StructField("latitude",     DoubleType(), True),
    StructField("longitude",    DoubleType(), True),
    StructField("zone",         StringType(), True),
    StructField("is_junction",  IntegerType(), True),
])
station_df = spark.createDataFrame(stations, stn_schema)

(station_df.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("setu_rail.silver.station_master"))

print("✅ silver.station_master written")

In [0]:
%sql
SELECT * FROM setu_rail.silver.station_master ORDER BY is_junction DESC, station_code;

## Step 3 — Enrich train timings with station + AQ features

In [0]:
%sql
-- silver.train_enriched: train timings joined with station master & avg AQ per city
CREATE OR REPLACE TABLE setu_rail.silver.train_enriched AS
WITH aq_city AS (
    SELECT city,
           AVG(pm25) AS pm25_avg,
           AVG(pm10) AS pm10_avg,
           AVG(no2)  AS no2_avg,
           AVG(so2)  AS so2_avg,
           AVG(co)   AS co_avg
    FROM   setu_rail.bronze.air_quality
    GROUP BY city
)
SELECT t.train_no,
       t.train_name,
       t.scheduled_hour,
       t.scheduled_time_raw,
       t.inferred_city        AS city,
       s.station_code,
       s.station_name,
       s.zone,
       s.is_junction,
       s.latitude,
       s.longitude,
       aq.pm25_avg,
       aq.pm10_avg,
       aq.no2_avg,
       aq.so2_avg,
       aq.co_avg,
       t.source_file
FROM   setu_rail.silver.train_timings t
LEFT JOIN setu_rail.silver.station_master s ON s.city = t.inferred_city
LEFT JOIN aq_city                        aq ON aq.city = t.inferred_city;

SELECT * FROM setu_rail.silver.train_enriched LIMIT 10;

✅ **Next:** `05_build_gold_features.ipynb`